# 33 · Графики ТО/ППР заказчика (2026): календарь плановых работ и сверка с сигналами

**Цель** — понять, можно ли использовать полученные от заказчика графики ППР АКМ и ТО/ТР АКМ и ДУ для (а) пометки «кампанийных» окон (как `data_utils.CAMPAIGN_WEEKS`) и (б) сверки с наблюдёнными всплесками сигналов 2026. Источник — `dataset/_external/maintenance_plan/` (read-only кэш).


In [1]:
import sys
import pathlib

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
import data_utils as du
import planned_work as pw

print("dataset:", du.find_dataset_dir())
print("ППР файл:", pw.PPR_FILE.exists(), "| ТО файл:", pw.TO_FILE.exists())


dataset: D:\Downloads_D\lct_a8\research\dataset
ППР файл: True | ТО файл: True


## 33.1 · Разбор графиков

Парсим обе выгрузки в нормированный вид и сохраняем артефакты в `dataset/`.


In [2]:
saved = pw.build_and_save()
for k, v in saved.items():
    print(f"  {k}: {v.name}")

ppr = pw.parse_ppr()
to = pw.parse_to()
print("\nППР: объектов =", ppr['объект'].nunique(), "| датчиков =", int(ppr['кол-во'].sum()))
print("ТО/ТР: объектов =", to['объект'].nunique(), "| отметок ТО/ТР =", len(to))
ppr[['объект','месяц_имя','кол-во','демонтаж','сдача']].head(10)


  ppr: planned_work_ppr_2026.csv
  to: planned_work_to_2026.csv
  weeks: planned_work_weeks_2026.csv

ППР: объектов = 26 | датчиков = 1008
ТО/ТР: объектов = 24 | отметок ТО/ТР = 546


,объект,месяц_имя,кол-во,демонтаж,сдача
0,Объект 1,Январь,56,2026-01-12,2026-01-27
1,Объект 2,None,13,NaT,2026-01-28
2,Объект 3,Февраль,62,2026-01-29,2026-02-12
3,Объект 4,None,47,2026-02-13,2026-02-27
4,Объект 5,Март,102,2026-03-02,2026-03-19
5,Объект 6,None,34,2026-03-23,2026-04-08
6,Объект 7,None,12,NaT,NaT
7,Объект 8,Апрель,101,2026-04-09,2026-04-22
8,Объект 9,None,40,2026-04-23,2026-05-14
9,Объект 10,Май,6,2026-05-15,2026-05-29


In [3]:
# ППР: сколько датчиков метана снимают по месяцам; ТО/ТР: сколько отметок по месяцам
mnames = {1:'янв',2:'фев',3:'мар',4:'апр',5:'май',6:'июн',7:'июл',8:'авг',9:'сен',10:'окт',11:'ноя',12:'дек'}
by_m_ppr = ppr.groupby('месяц')['кол-во'].sum().rename('датчиков_ППР')
by_m_to = to.groupby('месяц').size().rename('отметок_ТО')
by_m_tr = to[to['вид_работы'].eq('ТО+ТР')].groupby('месяц').size().rename('из_них_ТО+ТР')
m = pd.concat([by_m_ppr, by_m_to, by_m_tr], axis=1).fillna(0).astype(int)
m.index = [mnames[i] for i in m.index]
m


,датчиков_ППР,отметок_ТО,из_них_ТО+ТР
янв,69,55,11
фев,109,41,14
мар,148,35,6
апр,141,62,22
май,56,47,22
июн,33,33,4
июл,41,55,7
авг,143,41,8
сен,96,35,7
окт,62,62,27


## 33.2 · Календарь плановых работ в ISO-неделях 2026

Строим множество недель, в которые есть плановые работы, отдельно по источникам.


In [4]:
weeks = pw.planned_work_weeks(ppr, to)
wset = pw.planned_week_set(weeks)
all_2026 = set().union(*[set(pw.month_weeks(2026)[i]) for i in range(1, 13)])
print(f"Недель 2026 всего: {len(all_2026)}")
print(f"Недель с плановыми работами: {len(wset)} — покрытие {len(wset)/len(all_2026):.0%}")

# нагрузка по неделям: сколько объектов на ППР и сколько на ТО/ТР
load = weeks.groupby(['год_неделя','источник'])['объект'].nunique().unstack(fill_value=0)
load


Недель 2026 всего: 53
Недель с плановыми работами: 53 — покрытие 100%


источник,ППР_АКМ,ТО_ДУ_АКМ
год_неделя,,
202601,1,15
202602,1,15
202603,2,15
202604,2,15
202605,3,21
202606,1,9
202607,2,9
202608,1,9
202609,2,19


In [5]:
# Насыщенность: сколько объектов/записей приходится на каждую неделю
per_week = weeks.groupby('год_неделя').agg(объектов=('объект','nunique'),
                                           записей=('вид_работы','size'))
print('недель с записью о плановых работах:', len(per_week), 'из', len(all_2026))
print('медиана объектов/неделя:', per_week['объектов'].median())
print('минимум объектов/неделя:', per_week['объектов'].min())
print('\nРаспределение недель по числу объектов:')
print(per_week['объектов'].value_counts().sort_index().to_string())


недель с записью о плановых работах: 53 из 53
медиана объектов/неделя: 12.0
минимум объектов/неделя: 9

Распределение недель по числу объектов:
объектов
9      6
10     6
11    12
12     9
13     3
15     7
19     1
20     1
21     4
22     4


## 33.3 · Сверка с недельными сигналами 2026

Наложим календарь на `weekly_alarms_by_type.csv` (тревоги по типам датчиков) и проверим, совпадают ли всплески с плановыми окнами. Нас интересуют дым (пожарный риск) и газовый/метан.


In [6]:
wk = pd.read_csv(du.find_dataset_dir() / 'weekly_alarms_by_type.csv')
wk26 = wk[wk['год_неделя'].astype(str).str.startswith('2026')]
piv = wk26.pivot_table(index='год_неделя', columns='тип_датчика', values='тревог',
                       aggfunc='sum', fill_value=0)
smoke = piv['Датчик дыма'] if 'Датчик дыма' in piv.columns else None
gas = piv['Газовый датчик'] if 'Газовый датчик' in piv.columns else None
cmp = pd.DataFrame({'дым': smoke, 'газ': gas}).join(per_week)
cmp['плановые_недели'] = cmp.index.map(lambda w: 'да' if w in wset else 'нет')
cmp


,дым,газ,объектов,записей,плановые_недели
год_неделя,,,,,
202601,362,25,15,56,да
202602,136,8,15,56,да
202603,631,3,15,57,да
202604,603,30,15,57,да
202605,331,38,21,99,да
202606,642,74,9,42,да
202607,188,160,10,43,да
202608,603,1,10,42,да
202609,84,17,19,78,да


In [7]:
# Явно ли плановые работы объясняют крупные всплески дыма?
spikes = cmp[cmp['дым'] > cmp['дым'].median() + 3 * cmp['дым'].std()]
print('Недели-всплески дыма (> медиана+3sigma):')
print(spikes[['дым','объектов','записей']].to_string())
print('\nКорреляция числа объектов на неделе и тревог (дым):',
      round(cmp['объектов'].corr(cmp['дым']), 3))
print('Корреляция числа объектов на неделе и тревог (газ):',
      round(cmp['объектов'].corr(cmp['газ']), 3))


Недели-всплески дыма (> медиана+3sigma):
              дым  объектов  записей
год_неделя                          
202616      20187        11       63
202623      25212         9       34

Корреляция числа объектов на неделе и тревог (дым): -0.262
Корреляция числа объектов на неделе и тревог (газ): -0.319


In [8]:
# Сравнение с текущим календарём кампаний
from data_utils import CAMPAIGN_WEEKS
print('CAMPAIGN_WEEKS:')
for w1, w2, desc in CAMPAIGN_WEEKS:
    print(f'  {w1}..{w2}: {desc}')
camp = sorted(w for w in map(int, wk26['год_неделя'].astype(int).unique()) if du.week_is_campaign(w))
print('\nНедели 2026, помеченные как кампания:', camp)


CAMPAIGN_WEEKS:
  202116..202123: Переходный период до/после пика проверок 2021 (ППР-1479/СП 484-486)
  202117..202122: Пик кампании проверок дымовых извещателей апрель-май 2021
  202623..202623: Локальный всплеск 2026W23 — проверить природу

Недели 2026, помеченные как кампания: [202623]


## 33.4 · Выводы и ограничения

1. **Графики — это план на будущее (только 2026), не история.** Как supervised-цель использовать нельзя; только как календарь/контекст.
2. **Объекты анонимизированы как «Объект N» и не связаны с нашими `ид_объект`/каналом.** Без маппинга от заказчика per-object/per-channel маскировка невозможна.
3. **Глобальный недельный календарь насыщен** (плановые работы есть почти каждую неделю, т.к. распределены по 26 объектам) → как глобальный аналог `CAMPAIGN_WEEKS` он бесполезен. В отличие от кампании 2021 (одномоментной и затрагивавшей всех), здесь работы «размазаны».
4. **Графики касаются АКМ (метан) и ДУ, а не дымовых извещателей.** Крупные всплески дыма 2026 (W16, W23, W24) этими графиками не объясняются — природа всплесков иная (см. nb 06).
5. **Реальная ценность:** (а) синхронизация нормативного регламента с модулем превентивных заявок сервиса; (б) per-object маскировка окон ППР для задач sensor/gas — **после получения маппинга**; в) демонстрационная сверка «риск ML vs запланированное ТО».

**Нужно от заказчика:** маппинг «Объект N» ↔ `ид_объект` (диспетчерское название), и — при возможности — графики ТО/ППР за прошлые годы (для признаков по истории ремонтов).
